# URAE Tutorial: Formal, Multi-Valued & Modal Logic Systems

Welcome to the **Universal Rust Algebra Engine (URAE)** tutorial on formal logic systems.

$$A \to B = \neg A \lor B, \quad \Box A = \neg \Diamond \neg A, \quad \nu(A \to B) = \min(1, 1 - \nu(A) + \nu(B))$$

This notebook demonstrates:
1. **Classical Propositional Logic & DPLL SAT Solving**
2. **Three-Valued Logic Systems**: Kleene $K_3$, Łukasiewicz $Ł_3$, Bochvar nonsense logic, and Gödel-Dummett $G_3$ intuitionistic logic
3. **Modal Logic Operators**: Necessity ($\Box$) and Possibility ($\Diamond$) under Kripke frame semantics
4. **Assumptions & Automated Predicate Calculus Deduction**

In [1]:
import urae
from urae.logic import BoolExpr, SatSolver, ThreeValuedSystem, ThreeValuedValue, ThreeValuedLogic

# 1. Classical Propositional Logic & SAT Solving
p = BoolExpr.var(1)
q = BoolExpr.var(2)

# Tautology: p => (p \/ q)
tautology = BoolExpr.implies(p, BoolExpr.or_([p, q]))
print("Tautology Simplified:", tautology.simplify())
print("Is Satisfiable:", SatSolver.is_satisfiable(tautology))

# Contradiction: p /\ ~p
contradiction = BoolExpr.and_([p, BoolExpr.not_(p)])
print("Contradiction Simplified:", contradiction.simplify())
print("Is Satisfiable:", SatSolver.is_satisfiable(contradiction))

In [2]:
# 2. Comparing 3-Valued Logic Systems
# Systems: Kleene K3, Łukasiewicz Ł3, Bochvar, Gödel G3
u = ThreeValuedValue.Unknown
t = ThreeValuedValue.True_
f = ThreeValuedValue.False_

# In Kleene K3, u => u is Unknown (Identity law fails for indeterminate statements)
k3_impl = ThreeValuedLogic.implies(u, u, ThreeValuedSystem.KleeneK3)
print("Kleene K3: (u => u) =", k3_impl)

# In Łukasiewicz Ł3, u => u evaluates to True (Identity law holds)
l3_impl = ThreeValuedLogic.implies(u, u, ThreeValuedSystem.LukasiewiczL3)
print("Łukasiewicz Ł3: (u => u) =", l3_impl)

# In Bochvar logic, Unknown models a meaningless/fatal runtime error that poisons any connective:
bochvar_or = ThreeValuedLogic.or_(t, u, ThreeValuedSystem.Bochvar)
print("Bochvar: (True \/ Unknown) =", bochvar_or) # Unknown

# In Gödel G3 intuitionistic logic, negation of unknown is definitely False:
godel_not = ThreeValuedLogic.not_(u, ThreeValuedSystem.GodelG3)
print("Gödel G3: ¬(Unknown) =", godel_not) # False

In [3]:
# 3. Modal Operators: Box (Necessity) & Diamond (Possibility)
print("Box(True) =", ThreeValuedLogic.necessity(t))
print("Box(Unknown) =", ThreeValuedLogic.necessity(u))
print("Diamond(Unknown) =", ThreeValuedLogic.possibility(u))
print("Diamond(False) =", ThreeValuedLogic.possibility(f))

# Verification of Modal Duality: Diamond(A) == not Box(not A)
for val in [t, f, u]:
    dia_direct = ThreeValuedLogic.possibility(val)
    dia_dual = ThreeValuedLogic.not_(
        ThreeValuedLogic.necessity(ThreeValuedLogic.not_(val, ThreeValuedSystem.KleeneK3)),
        ThreeValuedSystem.KleeneK3
    )
    print(f"Duality holds for {val}: {dia_direct == dia_dual}")

In [4]:
# 4. Mathematical Predicate Assumptions & Contradiction Detection
ctx = urae.AssumptionsContext()
x_sym = urae.intern_symbol("x")

# When a symbol is asserted Positive, the engine automatically infers NonNegative, NonZero, and Real
ctx.assume(x_sym, urae.Predicate.Positive)
print("Is x Real?", ctx.is_(x_sym, urae.Predicate.Real))
print("Is x NonZero?", ctx.is_(x_sym, urae.Predicate.NonZero))
print("Is assumptions set consistent?", ctx.is_consistent())

# Introduce a contradictory assumption: x is Negative
ctx.assume(x_sym, urae.Predicate.Negative)
print("After assuming Negative, is consistent?", ctx.is_consistent()) # False